In [1]:
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
from bioio import BioImage
import os
import numpy as np
from dask.distributed import Client, get_client
import napari
import xmltodict

from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion
from skimage import io
import glob
from pathlib import Path

In [2]:
viewer = napari.Viewer()

In [ ]:
try:
  client = get_client()
except ValueError:
  client = Client()

print(client.dashboard_link)

http://127.0.0.1:8787/status


2026-04-21 11:14:50,860 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "c:\Users\Leica-User\miniforge3\envs\autoscape\Lib\site-packages\tornado\websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Leica-User\miniforge3\envs\autoscape\Lib\site-packages\tornado\web.py", line 3388, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Leica-User\miniforge3\envs\autoscape\Lib\site-packages\bokeh\server\views\ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exc

In [44]:
root = r'E:\UserData\Johannes\20260420\2026_04_21_09_49_52--TRG1970_001\TileScan 1'
file_names = glob.glob(os.path.join(root, "**", "*ome.tif"), recursive=True)
file_names

['E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\10.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\3.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\4.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\5.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\6.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\7.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\8.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\C\\9.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\D\\10.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_21_09_49_52--TRG1970_001\\TileScan 1\\D\\3.ome.tif',
 'E:\\UserData\\Johannes\\20

In [45]:
def find_in_nested_dict(data, key, value):
    """
    Recursively search for objects containing a specific key-value pair.
    Returns a list of all matching objects.
    """
    results = []
    
    def search(obj):
        if isinstance(obj, dict):
            if obj.get(key) == value:
                results.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(data)
    return results  

In [ ]:
write_jobs = []
images = []
transforms = []
for filename in file_names:
	img_basename = Path(filename).stem.split(".")[0] 
	outfile_oz = img_basename + ".ome.zarr"
	bf = BioImage(filename)
	

	if bf.ome_metadata.images[0].pixels.physical_size_z is None:
		size_z = 0
	else:
		size_z = bf.ome_metadata.images[0].pixels.physical_size_z

	scale = {
		"z": size_z,
		"y": bf.ome_metadata.images[0].pixels.physical_size_y,
		"x": bf.ome_metadata.images[0].pixels.physical_size_x
	}

	axes_units = {
		"z": "micrometer",
		"y": "micrometer",
		"x": "micrometer",
	}
	
	# parse tile metadata
	metadata_file = os.path.join(Path(filename).parent, "Metadata", f"{img_basename}.xlif")
	metadata_dict = xmltodict.parse(open(metadata_file).read())
	tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

	# n_series= bf.series_count()
	n_series = len(bf.scenes)

	all_positions = []
	if n_series > 1:
		sims = []
		for idx in range(n_series):
			bf.set_scene(bf.scenes[idx])
			dask_image = bf.dask_data
			size_tile = np.asarray(dask_image.squeeze().shape) * np.asarray([scale["z"], scale["y"], scale["x"]])
			# dask_image = bf.to_dask(series=idx)
			ts_info = tile_scan_info[idx]

			t_z = float(ts_info["@PosZ"])
			t_y = float(ts_info["@PosY"])
			t_x = float(ts_info["@PosX"])

			all_positions.append(np.asarray([t_z, t_y, t_x]))

			sims.append(
				si_utils.get_sim_from_array(
					dask_image,
					dims=["t", "c", "z", "y", "x"],
					scale=scale,
					translation={"z": t_z, "y": t_y, "x": t_x},
					transform_key="stage_metadata",
					)
			)

		image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion)
	else:
		# image = bf.to_dask(series=0)
		image = bf.dask_data
		size_tile = np.asarray(dask_image.squeeze()) * np.asarray([scale["z"], scale["y"], scale["x"]])
		t_z = float(tile_scan_info["@PosZ"])
		t_y = float(tile_scan_info["@PosY"])
		t_x = float(tile_scan_info["@PosX"])

		all_positions.append(np.asarray([t_z, t_y, t_x]))

	all_positions = np.stack(all_positions, axis=0)
	position_corner = np.min(all_positions, axis=0) - size_tile / 2

	ngff_image = NgffImage(
		image.squeeze(),
		dims=["z", "y", "x"],
		scale=scale,
		axes_units=axes_units,
		name=Path(filename).parent.stem + img_basename
	)
	ngff_multiscales = NgffMultiscales(
		image=ngff_image,
		scale_factors=[
		{"z": 2, "y": 2, "x": 2},
		{"z": 4, "y": 4, "x": 4},
		{"z": 8, "y": 8, "x": 8},
		{"z": 16, "y": 16, "x": 16},
		],
	)
	images.append(ngff_multiscales)

	transform = Translation(
		translation=position_corner.tolist(),
		input=CoordinateSystemIdentifier(
			path=ngff_multiscales.name,
			name=ngff_multiscales.metadata.coordinateSystems[0].name
		),
		output=CoordinateSystemIdentifier(
			name="world"
		)
	)
	transforms.append(transform)

<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small to downsample further.
<string>:4: UserWarning: Dimension z is too small t

In [ ]:
cs_world = CoordinateSystem(
	name="world",
	axes=(
		Axis(name="z", type="space", unit="micrometer"),
		Axis(name="y", type="space", unit="micrometer"),
		Axis(name="x", type="space", unit="micrometer"),
	)
)

ngff_scene = NgffScene(
	images=images,
	transformations=transforms,
	coordinate_systems=(
		cs_world,
	)
)

delayed_objects = ngff_scene.to_ome_zarr(
	os.path.join(root, "scene.ome.zarr")
)

KeyboardInterrupt: 

## Show scenes



In [39]:
viewer.open(os.path.join(root, "scene.ome.zarr"), plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': '9'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.041731425, 0.0797631708]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}
Scene.matches {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': '9'}, 'output': {'name': 'world'}, 'translation': [0.00278575387, 0.041731425, 0.0797631708]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}
Scene
spec <napari_ome_zarr.ome_zarr_reader.Scene object at 0x000001DBCF38F980>
Scene.iter_nodes...fin

[<Image layer 'scene.ome [1]' at 0x1db8ad30800>]

In [40]:
viewer.layers['Points'].data * viewer.layers['Points'].scale

array([[0.00281915, 0.04430795, 0.08423483]])

In [42]:
viewer.layers['scene.ome [1]'].affine.scale

array([1.66992200e-05, 4.79765625e-07, 4.79843750e-07])

In [18]:
layer = viewer.layers['scene.ome [1]']
max_projection = layer.data[0].min(axis=0)
viewer.add_image(max_projection, scale = layer.affine.scale[1:], translate=layer.affine.translate[1:], name="max projection")

<Image layer 'max projection [1]' at 0x14656604890>

In [ ]:
for layer in viewer.layers:
	projection = layer.data[0].min(axis=0).rechunk(640, 640)

	# iterate over chunks and save as individual images
	for idx in np.ndindex(projection.numblocks):
		chunk = projection.blocks[idx]
		if not chunk.shape == (640, 640):
			continue

		client.submit(io.imsave, os.path.join(r"C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20260317\projections", f"{layer.name}_chunk_{idx}.tif"), chunk)

In [46]:
viewer_projections.layers.clear()
viewer_projections.add_image(chunk, contrast_limits=(p5.compute(), p95.compute()))

<Image layer 'chunk' at 0x1d8aa80ea20>

In [ ]:
viewer.layers[0].data[0].min(axis=0).rechunk(640, 640)

(4, 5)

In [35]:
p5.compute()

np.float64(243.0)

In [36]:
p95.compute()

np.float64(981.0)